# Experiment: Config C (QLoRA FT) & Config D (Base Model) Inference

This notebook runs on **Colab Pro with T4 or A100 GPU** to generate podcast transcripts
using local Llama-3.1-8B models (with and without QLoRA fine-tuning).

**Prerequisites:**
- Upload `podcast-expert-lora.zip` (QLoRA adapter weights — exported from training notebook)
- Upload 9 `*.txt` files from `experiments/evaluation/data/topics/`

**Outputs:**
- `baseline_ft/*.json` — Config C (QLoRA-Llama-3.1-8B fine-tuned)
- `baseline_base/*.json` — Config D (Llama-3.1-8B vanilla base)
- Each JSON contains 4 system metrics: latency, tokens, cost, GPU memory

**Note:** Run Config D first (Cell 4), then Config C (Cell 5).
GPU memory is freed between the two runs via `del model` + `torch.cuda.empty_cache()`.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers

In [ ]:
import json
import os
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import torch
from google.colab import files

# ---- Upload adapter weights ----
if not Path("podcast-expert-lora/adapter_config.json").exists():
    print("Upload podcast-expert-lora.zip:")
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith(".zip"):
            with zipfile.ZipFile(fname) as zf:
                zf.extractall(".")
            print(f"Extracted {fname}")

# ---- Upload source documents ----
topics_dir = Path("topics")
if not topics_dir.exists() or len(list(topics_dir.glob("*.txt"))) < 9:
    topics_dir.mkdir(exist_ok=True)
    print("Upload 9 topic .txt files (select all at once):")
    uploaded = files.upload()
    for fname, content in uploaded.items():
        (topics_dir / fname).write_bytes(content)
    print(f"Uploaded {len(uploaded)} files to topics/")

print(f"\nAdapter files: {list(Path('podcast-expert-lora').glob('*'))}")
print(f"Topic files:   {sorted(topics_dir.glob('*.txt'))}")

In [ ]:
# ============================================================
# Topic definitions & generation utilities
# ============================================================

TOPICS = [
    {"topic_id": "tech_1_transformer_attention", "topic": "Transformer Attention Mechanism", "domain": "technology"},
    {"topic_id": "tech_2_qlora_finetuning", "topic": "QLoRA Efficient Fine-tuning", "domain": "technology"},
    {"topic_id": "tech_3_multi_agent", "topic": "LLM Multi-Agent Architecture", "domain": "technology"},
    {"topic_id": "hum_1_ai_bias_fairness", "topic": "AI Bias and Fairness", "domain": "humanities"},
    {"topic_id": "hum_2_ai_privacy", "topic": "AI Privacy and Data Protection", "domain": "humanities"},
    {"topic_id": "hum_3_ai_safety_alignment", "topic": "AI Safety and Alignment", "domain": "humanities"},
    {"topic_id": "med_1_ai_medical_imaging", "topic": "AI-Assisted Medical Imaging", "domain": "medicine"},
    {"topic_id": "med_2_clinical_trials", "topic": "Clinical Trial Design", "domain": "medicine"},
    {"topic_id": "med_3_ai_drug_discovery", "topic": "AI in Drug Discovery", "domain": "medicine"},
]

BASELINE_PROMPT = (
    "You are a podcast script writer. Given source material, write a natural "
    "two-person podcast conversation between a Host and an Expert.\n\n"
    "Topic: {topic}\n\n"
    "Source material:\n{source_text}\n\n"
    "Requirements:\n"
    "- Generate exactly 10 turns (5 host lines + 5 expert lines, strictly alternating, starting with host).\n"
    "- The Host asks engaging questions and guides the conversation.\n"
    "- The Expert provides insightful, accurate answers grounded in the source material.\n"
    "- Each turn: 1-3 sentences, natural and suitable for audio.\n\n"
    "Return ONLY a JSON array (no markdown fences). Each element: "
    '{"speaker": "host" or "expert", "text": "..."}'
)

MAX_NEW_TOKENS = 1500
TEMPERATURE = 0.7
TOP_P = 0.9


def load_source(topic_id: str) -> str:
    path = Path("topics") / f"{topic_id}.txt"
    return path.read_text(encoding="utf-8")


def parse_turns(text: str) -> list:
    import re
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```\w*\n?", "", cleaned)
        cleaned = re.sub(r"\n?```$", "", cleaned)
        cleaned = cleaned.strip()
    try:
        data = json.loads(cleaned)
        if isinstance(data, list):
            return data
    except json.JSONDecodeError:
        pass
    match = re.search(r"\[.*\]", cleaned, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return [{"speaker": "host", "text": cleaned}]


def generate_baseline(model, tokenizer, topic: dict, source_text: str) -> dict:
    """Generate a full podcast conversation with a single prompt.

    System prompt format matches the SFT training data so the fine-tuned
    model receives familiar context (source material in system role).
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a podcast script writer. Given source material, write a natural "
                "two-person podcast conversation between a Host and an Expert.\n\n"
                f"Topic: {topic['topic']}\n\n"
                f"Source material:\n{source_text[:4000]}"
            ),
        },
        {
            "role": "user",
            "content": (
                "Requirements:\n"
                "- Generate exactly 10 turns (5 host lines + 5 expert lines, strictly alternating, starting with host).\n"
                "- The Host asks engaging questions and guides the conversation.\n"
                "- The Expert provides insightful, accurate answers grounded in the source material.\n"
                "- Each turn: 1-3 sentences, natural and suitable for audio.\n\n"
                "Return ONLY a JSON array (no markdown fences). "
                'Each element: {"speaker": "host" or "expert", "text": "..."}'
            ),
        },
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    input_ids = inputs["input_ids"]
    input_len = input_ids.shape[-1]

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
        )

    latency = time.time() - t0
    peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 3)

    output_ids = outputs[0][input_len:]
    output_len = len(output_ids)
    response_text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    turns = parse_turns(response_text)

    return {
        "turns": turns,
        "metrics": {
            "latency_seconds": round(latency, 2),
            "input_tokens": int(input_len),
            "output_tokens": int(output_len),
            "total_tokens": int(input_len + output_len),
            "estimated_cost_usd": 0.0,
            "peak_gpu_memory_gb": round(peak_mem, 3),
        },
    }


def run_config(model, tokenizer, config_name: str, model_label: str, out_dir: str):
    """Run baseline generation for all 9 topics."""
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 50}")
    print(f"Config {config_name} — {model_label}")
    print(f"{'=' * 50}")

    for i, topic in enumerate(TOPICS, 1):
        json_path = out_path / f"{topic['topic_id']}.json"
        if json_path.exists():
            print(f"  [{i}/9] SKIP {topic['topic_id']} (exists)")
            continue

        source_text = load_source(topic["topic_id"])
        print(f"  [{i}/9] {topic['topic_id']} ...", end=" ", flush=True)

        result = generate_baseline(model, tokenizer, topic, source_text)

        full_result = {
            "config": config_name,
            "topic_id": topic["topic_id"],
            "topic": topic["topic"],
            "domain": topic["domain"],
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "model": model_label,
            "turns": result["turns"],
            "metrics": result["metrics"],
        }

        json_path.write_text(json.dumps(full_result, indent=2, ensure_ascii=False))
        m = result["metrics"]
        print(
            f"done  latency={m['latency_seconds']:.1f}s  "
            f"tokens={m['total_tokens']}  "
            f"gpu_mem={m['peak_gpu_memory_gb']:.2f}GB"
        )

    print(f"\nResults saved to {out_path}/")

In [ ]:
# ============================================================
# Config D — Base Llama-3.1-8B-Instruct (no adapter)
# ============================================================

from unsloth import FastLanguageModel

print("Loading base model (no adapter)...")
model_d, tokenizer_d = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.1-8B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(model_d)
print("Base model loaded.\n")

run_config(model_d, tokenizer_d, "D", "Llama-3.1-8B-Instruct", "baseline_base")

# Free GPU memory before loading next model
del model_d, tokenizer_d
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Config C — QLoRA Fine-tuned Llama-3.2-1B
# ============================================================

from unsloth import FastLanguageModel

print("Loading QLoRA adapter...")
model_c, tokenizer_c = FastLanguageModel.from_pretrained(
    model_name="podcast-expert-lora",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(model_c)
print("QLoRA model loaded.\n")

run_config(model_c, tokenizer_c, "C", "QLoRA-Llama-3.1-8B", "baseline_ft")

del model_c, tokenizer_c
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Download results
# ============================================================

import shutil

archive_name = "experiment_local_results"
shutil.make_archive(archive_name, "zip", ".", "baseline_ft")

# Append baseline_base to the same zip
import zipfile
with zipfile.ZipFile(f"{archive_name}.zip", "a") as zf:
    for f in Path("baseline_base").glob("*.json"):
        zf.write(f)

files.download(f"{archive_name}.zip")
print(f"\nDownload started: {archive_name}.zip")
print("Extract baseline_ft/ and baseline_base/ into experiments/evaluation/data/")